In [0]:
%run ../../common/_utils_config

In [0]:
%run ../../common/_utils_reference_data

In [0]:
%run ../utils/_utils_silver_contracts

In [0]:
%run ../utils/_utils_silver_mappings

In [0]:
%run ../utils/_utils_silver_parser

In [0]:
%run ../utils/_utils_silver_quality

In [0]:
failures = []


def check(description, condition):
    print(("✅ " if condition else "❌ ") + description)
    if not condition:
        failures.append(description)


def failed_count(result, rule, column):
    return next((m["failed"] for m in result["metrics"] if m["rule"] == rule and m["column"] == column), None)


contracts = load_all_entity_contracts()
mappings = load_all_mappings()

In [0]:
location_df = build_reference_df(contracts["location"], contracts)
data_source_df = build_reference_df(contracts["data_source"], contracts)
display(location_df.limit(5))
display(data_source_df)

check("location has 52 rows", location_df.count() == 52)
check("location PK is unique", location_df.select("PKLocation").distinct().count() == 52)
malaga = location_df.filter(F.col("province_code") == "29").first()
check("PKLocation = md5(province_code) (Python md5)", malaga["PKLocation"] == key_hash_python(["29"]))
check("location FKDataSource = md5('locations_es')", malaga["FKDataSource"] == key_hash_python(["locations_es"]))
check("location columns are typed (latitude double)", dict(location_df.dtypes)["latitude"] == "double")
check("data_source contains ree and open_meteo_archive",
      {"ree", "open_meteo_archive"} <= {r["source_name"] for r in data_source_df.collect()})

In [0]:
ree_parsed, _, ree_quality = dry_run_with_quality(mappings["ree__electricity_price"], contracts)
print_quality_report(ree_quality)
weather_parsed, _, weather_quality = dry_run_with_quality(
    mappings["open_meteo_archive__weather_observation_hourly"], contracts)
print_quality_report(weather_quality)

for result in (ree_quality, weather_quality):
    check(f"{result['mapping']}: not failed (status {result['status']})", result["status"] in ("ok", "warning"))
    check(f"{result['mapping']}: no rows quarantined", result["quarantined_rows"] == 0)
    check(f"{result['mapping']}: every FK exists in its reference table",
          all(m["failed"] == 0 for m in result["metrics"] if m["rule"] == "reference_exists"))

In [0]:
weather_mapping = mappings["open_meteo_archive__weather_observation_hourly"]
base = weather_parsed.filter(F.col("province_code") == "29")
base_count = base.count()
one = base.orderBy("datetime_utc").limit(1)

dirty = (one.withColumn("temperature_c", F.lit(99.0))                                         # range
         .unionByName(one.withColumns({"datetime_utc": F.lit(None).cast("timestamp"),         # not_null
                                       f"{raw_prefix}datetime_utc": F.lit(None).cast("string")}))
         .unionByName(one.withColumns({"province_code": F.lit("99"),                          # reference_exists
                                       "FKLocation": F.md5(F.lit("99"))}))
         .unionByName(one.withColumns({"temperature_c": F.lit(None).cast("double"),           # conversion
                                       f"{raw_prefix}temperature_c": F.lit("abc")})))
test_df = base.unionByName(dirty).unionByName(one)                                            # duplicate_pk

result = evaluate_quality(test_df, weather_mapping, contracts)
print_quality_report(result)

check("4 dirty rows quarantined", result["quarantined_rows"] == 4)
check("valid rows = original rows (duplicate removed)", result["valid_rows"] == base_count)
check("range failure counted", failed_count(result, "range", "temperature_c") == 1)
check("null business key counted", failed_count(result, "not_null", "datetime_utc") == 1)
check("missing location counted", failed_count(result, "reference_exists", "FKLocation") == 1)
check("failed conversion counted", failed_count(result, "conversion", "temperature_c") == 1)
check("duplicated PK counted (warn)", failed_count(result, "duplicate_pk", "PKWeatherObservationHourly") == 1)
check("status is warning (below the quarantine limit)", result["status"] == "warning")

rules_in_quarantine = {rule for r in result["quarantine"].select("failed_rules").collect() for rule in r[0]}
check("quarantine keeps the failed rule names",
      {"range:temperature_c", "not_null:datetime_utc", "reference_exists:FKLocation",
       "conversion:temperature_c"} <= rules_in_quarantine)
conversion_row = result["quarantine"].filter(F.array_contains("failed_rules", "conversion:temperature_c")).first()
check("quarantine keeps the raw value that failed", conversion_row["raw_values"]["temperature_c"] == "abc")

In [0]:
days = sorted(r[0] for r in base.select(F.to_date("datetime_utc")).distinct().collect())
middle_day = days[len(days) // 2]
holed = base.filter(~((F.to_date("datetime_utc") == F.lit(middle_day)) & (F.hour("datetime_utc") == 12)))
holed_result = evaluate_quality(holed, weather_mapping, contracts)
print_quality_report(holed_result)
check(f"missing hour on {middle_day} detected by completeness",
      failed_count(holed_result, "completeness", "datetime_utc") == 1)
check("completeness issue shows 23 of 24 rows",
      any(i["count"] == 23 and i["expected"] == 24 for i in holed_result["completeness_issues"]))

broken_batch = base.withColumn("temperature_c", F.lit(99.0))
broken_result = evaluate_quality(broken_batch, weather_mapping, contracts)
print_quality_report(broken_result)
check("a batch with 100% quarantined rows gets status fail", broken_result["status"] == "fail")

In [0]:
if failures:
    raise AssertionError(f"{len(failures)} check(s) failed: {failures}")
print("✅ All silver quality checks passed")